In [3]:
import os

In [1]:
%pwd

'd:\\Text-summerizer\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'd:\\Text-summerizer'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)

class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path 

In [10]:
import importlib
import textsummarizer.utils.common

importlib.reload(textsummarizer.utils.common)

<module 'textsummarizer.utils.common' from 'D:\\Text-summerizer\\src\\textsummarizer\\utils\\common.py'>

In [11]:
from textsummarizer.constants import *

from textsummarizer.utils.common import read_yaml, create_directories

In [14]:
import importlib
import textsummarizer.constants

importlib.reload(textsummarizer.constants)

from textsummarizer.constants import *

In [15]:
class configurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=CONFIG_FILE_NAME):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion_config
        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        return data_ingestion_config

In [16]:
import os
import urllib.request as request
import zipfile
from textsummarizer.utils.common import get_size
from textsummarizer.logging import logger

In [17]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")


    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"File extracted at: {unzip_path} with size: {get_size(Path(unzip_path))}")

In [18]:
try:
    config = configurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    logger.exception(e)

[2026-09-13 22:24:40,363]:INFO:textsummarizerLogger: yaml file: config\config.yaml loaded successfully
[2026-09-13 22:24:40,375]:INFO:textsummarizerLogger: yaml file: params.yaml loaded successfully
[2026-09-13 22:24:41,468]:INFO:textsummarizerLogger: artifacts\data_ingestion\data.zip downloaded with following info: 
Date: Sun, 13 Sep 2026 16:54:41 GMT
Content-Type: text/html; charset=utf-8
x-repository-download: git clone https://github.com/Varunkumarke/data-tutorials.git
x-raw-download: https://raw.githubusercontent.com/Varunkumarke/data-tutorials/main/samsumdata.zip
Vary: X-PJAX, X-PJAX-Container, Turbo-Visit, Turbo-Frame, X-Requested-With, X-GitHub-Client-Version, Sec-Fetch-Site,Accept-Encoding, Accept, X-Requested-With
ETag: W/"4b9028d1db32e34c0345467e157a44ec"
Cache-Control: max-age=0, private, must-revalidate
Strict-Transport-Security: max-age=31536000; includeSubdomains; preload
X-Frame-Options: deny
X-Content-Type-Options: nosniff
X-XSS-Protection: 0
Referrer-Policy: no-referr